In [ ]:
import torch
from ultralytics import YOLO
import ultralytics.nn.tasks as tasks
from spdconv import SPDConv   # ya da senin dosya adın neyse
tasks.SPDConv = SPDConv

In [ ]:
device = 0 if torch.cuda.is_available() else "cpu" 
print(f"Kullanilan cihaz: {'GPU (cuda:0)' if device == 0 else 'CPU'}")
if device == "cpu":
        print("UYARI: GPU bulunamadi, egitim CPU'da yapilacak ve cok yavas olabilir.")

# --- Veri seti yolu ---
# Once depo icindeki dataset/ klasoruna bakar (README'deki kurulum),
# bulamazsa gelistirme makinesindeki konuma duser.
import os

_CANDIDATES = [
    "../dataset/data.yaml",
    "../../../dataset2/yolo26/YOLO.v1i.yolo26/data.yaml",
]
DATA = next((p for p in _CANDIDATES if os.path.exists(p)), _CANDIDATES[0])
print("veri seti:", os.path.abspath(DATA), "->", os.path.exists(DATA))

In [ ]:
# 1) P2 head'li YOLO26 mimarisini (henüz eğitilmemiş yapı) oluştur
#    "s" ölçeğini baseline ile aynı tut ki ağırlık transferi daha sağlıklı olsun
model = YOLO("spd-covn.yaml").load("../baseline/runs/detect/train/weights/best.pt")
  # kendi baseline yolunu yaz

# 2) Baseline eğitiminden çıkan ağırlıkları yükle
#    .load() ismi/shape'i eşleşen katmanları (backbone, P3-P4-P5) otomatik aktarır,
#    yeni eklenen P2 katmanları rastgele başlatılmış olarak kalır
#model.load("runs/detect/train/weights/best.pt")  # kendi baseline yolunu yaz

In [ ]:
# --- Aşama A: Backbone dondurulmuş, yeni katmanlar + head ısınıyor ---
model.train(
    data=DATA,
    epochs=500,
    imgsz=640,
    batch=16,
    optimizer="SGD",
    device=device,
    resume="True",
    patience=100,
    name="ablation_spd_conv_v",
)

In [ ]:
import csv, glob
run = sorted(glob.glob("runs/detect/ablation_spd_conv_v2/results.csv"))[-1]
rows = list(csv.DictReader(open(run)))
print(run, "| epoch:", len(rows))
for r in rows[:2] + rows[-3:]:
    print("ep %-4s box=%-8s cls=%-8s mAP50=%-7s mAP50-95=%s" % (
        r["epoch"], r["train/box_loss"], r["train/cls_loss"],
        r["metrics/mAP50(B)"], r["metrics/mAP50-95(B)"]))

b = max(rows, key=lambda r: float(r["metrics/mAP50-95(B)"]))
print("\nEN IYI epoch %s -> P %.3f R %.3f mAP50 %.4f mAP50-95 %.4f" % (
    b["epoch"], float(b["metrics/precision(B)"]), float(b["metrics/recall(B)"]),
    float(b["metrics/mAP50(B)"]), float(b["metrics/mAP50-95(B)"])))

box1 = float(rows[0]["train/box_loss"])
print("\n[saglik] 1. epoch train/box_loss = %.2f -> %s" % (
    box1, "OK" if box1 < 20 else "!! KAYIP OLCEGI PATLAMIS, DURDUR"))
if len(rows) < 500:
    print("[uyari] kosu %d epoch'ta bitmis, 500 degil - yarida kesilmis olabilir" % len(rows))

In [ ]:
from pathlib import Path
best = Path(sorted(glob.glob("runs/detect/ablation_spd_conv_v2/weights/best.pt"))[-1])
print("test ediliyor:", best)

res = YOLO(str(best)).val(
    data=DATA,
    split="test",
    imgsz=640,
    device=device,
)
d = res.results_dict
print("\n=== TEST ===")
for k, lbl in [("metrics/precision(B)", "Precision"),
               ("metrics/recall(B)", "Recall"),
               ("metrics/mAP50(B)", "mAP@0.5"),
               ("metrics/mAP50-95(B)", "mAP@0.5:0.95")]:
    v = d.get(k)
    print(f"{lbl:>14}: {v*100:.2f}%" if v is not None else f"{lbl:>14}: yok")

try:
    for i, c in enumerate(res.names.values()):
        print(f"  {c:>6}: mAP50={res.box.ap50[i]:.4f}  mAP50-95={res.box.ap[i]:.4f}")
except Exception as e:
    print("sinif bazinda metrik alinamadi:", e)